In [17]:

import os
import hashlib
from collections import Counter
from typing import List, Union
import numpy as np
import cv2
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import torchvision.models as models
from torchvision.ops import sigmoid_focal_loss
import tqdm
from sklearn.metrics import classification_report
from google.colab import drive
import subprocess



# Mount Google Drive
try:
    drive.mount('/content/drive')
    print("Google Drive mounted successfully.")
except ImportError:
    print("Not running in Google Colab. Skipping Drive mount.")

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Google Drive mounted successfully.
Using device: cuda
GPU Name: Tesla T4


In [18]:

# --- Paths ---
subprocess.run(["unzip", 
    "-q", "/content/drive/MyDrive/Datasets/kaggle_knee_osteoarthritis.zip",
    "-d", "/content/Datasets"])
#  unzip -q /content/drive/MyDrive/Datasets/kaggle_knee_osteoarthritis.zip -d /content/Datasets
DATASET_ROOT_PATH = "/content/Datasets/kaggle_knee_osteoarthritis"
CHECKPOINT_SAVE_DIR = "/content/drive/MyDrive/Models/efficientnet_b4_checkpoints"

os.makedirs(CHECKPOINT_SAVE_DIR, exist_ok=True)

# --- Training Hyperparameters ---
EPOCHS = 50
EPOCHS_STAGE1 = 20  # Max epochs for Stage 1 (Classifier only)
BATCH_SIZE = 16
IMG_SIZE = 380  # EfficientNet-B4 uses 380 for higher resolution
INITIAL_LR = 1e-4

# --- Fine-Tuning & Loss Strategy Options ---
FINE_TUNE = True               # True to use Discriminative Fine-Tuning (3 groups) for EfficientNet
USE_FOCAL_LOSS = False         # True to use Sigmoid Focal Loss
EARLY_STOPPING_PATIENCE = 10   # Set to 0 to disable early stopping

# --- Ordinal Classification Type Options ---
# "none"             -> Standard Cross Entropy (5 classes)
# "expected_value"   -> Cross Entropy/Focal Loss + Expected Value Regularization (5 classes)
# "threshold"        -> Binary Cross Entropy with Logits (Frank-Hall Threshold, 4 classes)
ORDINAL_TYPE = "expected_value"


In [19]:

class SquarePadOpenCV(object):
    """Pads a rectangular image to a square."""
    def __call__(self, image):
        h, w = image.shape[:2]
        max_wh = max(h, w)
        pad_top = (max_wh - h) // 2
        pad_bottom = max_wh - h - pad_top
        pad_left = (max_wh - w) // 2
        pad_right = max_wh - w - pad_left
        
        padded_image = cv2.copyMakeBorder(
            image, pad_top, pad_bottom, pad_left, pad_right, 
            borderType=cv2.BORDER_CONSTANT, value=[0, 0, 0]
        )
        return padded_image

class OpenCVCLAHE(object):
    """Applies CLAHE (Contrast Limited Adaptive Histogram Equalization) using OpenCV."""
    def __init__(self, clip_limit=2.0, tile_grid_size=(8, 8)):
        self.clip_limit = clip_limit
        self.tile_grid_size = tile_grid_size

    def __call__(self, img_rgb: np.ndarray) -> np.ndarray:
        clahe = cv2.createCLAHE(clipLimit=self.clip_limit, tileGridSize=self.tile_grid_size)
        img_lab = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2LAB)
        l_channel, a_channel, b_channel = cv2.split(img_lab)
        clahe_l_channel = clahe.apply(l_channel)
        merged_lab_image = cv2.merge((clahe_l_channel, a_channel, b_channel))
        return cv2.cvtColor(merged_lab_image, cv2.COLOR_LAB2RGB)

def get_transforms(img_size=224):
    """Returns training and validation transforms."""
    train_transform = transforms.Compose([
        SquarePadOpenCV(),
        OpenCVCLAHE(),
        transforms.ToPILImage(),
        transforms.RandomAffine(degrees=3, translate=(0.02, 0.02), scale=(0.95, 1.05), shear=2),
        transforms.ColorJitter(brightness=0.03, contrast=0.03),
        transforms.Resize((img_size, img_size)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    
    val_transform = transforms.Compose([
        SquarePadOpenCV(),
        OpenCVCLAHE(),
        transforms.ToPILImage(),
        transforms.Resize((img_size, img_size)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    return train_transform, val_transform

def remove_duplicate_images(image_paths: List[str], labels: List[int], exclude_hashes: set = None, categories: List[str] = None):
    """Removes duplicate images using MD5 hashing."""
    total_found = len(image_paths)
    unique_paths, unique_labels, unique_hashes = [], [], set()
    internal_dup_count, leakage_count = 0, 0
    
    for path, label in zip(image_paths, labels):
        hash_md5 = hashlib.md5()
        try:
            with open(path, "rb") as f:
                for chunk in iter(lambda: f.read(4096), b""):
                    hash_md5.update(chunk)
            h = hash_md5.hexdigest()
        except Exception as e:
            print(f"Warning: Could not read image {path}: {e}")
            continue
            
        if exclude_hashes and h in exclude_hashes:
            leakage_count += 1
            continue
        if h in unique_hashes:
            internal_dup_count += 1
            continue
            
        unique_hashes.add(h)
        unique_paths.append(path)
        unique_labels.append(label)
        
    class_counts = Counter(unique_labels)
    print(f"\n--- Dataset Statistics & Deduplication ---")
    print(f"  - Total files: {total_found} | Unique kept: {len(unique_paths)}")
    print(f"  - Internal dupes removed: {internal_dup_count} | Cross-split leaks removed: {leakage_count}")
    return unique_paths, unique_labels, unique_hashes

class KaggleKneeOsteoarthritisDataset(Dataset):
    """Dataset class specifically for the Kaggle Knee Osteoarthritis dataset."""
    def __init__(self, root: str, split_dir: str, transform=None, exclude_hashes: set = None):
        self.root = root
        self.transform = transform
        self.exclude_hashes = exclude_hashes
        raw_paths, raw_labels = [], []
        split_path = os.path.join(root, split_dir)
        
        if not os.path.isdir(split_path): 
            raise FileNotFoundError(f"Split directory not found: {split_path}")
            
        class_names = sorted([d for d in os.listdir(split_path) if os.path.isdir(os.path.join(split_path, d)) and d.isdigit()])
        print(f"Loading '{split_dir}' split from: {split_path}")
        
        for class_name in class_names:
            class_dir = os.path.join(split_path, class_name)
            label = int(class_name)
            valid_extensions = ('.png', '.jpg', '.jpeg')
            image_files = [f for f in os.listdir(class_dir) if f.lower().endswith(valid_extensions)]
            for file_name in image_files:
                raw_paths.append(os.path.join(class_dir, file_name))
                raw_labels.append(label)
                
        self.image_paths, self.labels, self.image_hashes = remove_duplicate_images(
            raw_paths, raw_labels, exclude_hashes=self.exclude_hashes, categories=class_names
        )

    def load_image_from_path(self, image_path: str) -> np.ndarray:
        img_bgr = cv2.imread(image_path)
        if img_bgr is None: raise IOError(f"Could not read image: {image_path}")
        return cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

    def __getitem__(self, idx: int):
        image = self.load_image_from_path(self.image_paths[idx])
        label = self.labels[idx]
        if self.transform: image = self.transform(image)
        return image, label

    def __len__(self) -> int: 
        return len(self.image_paths)


In [20]:

class EfficientNetB4Model(nn.Module):
    def __init__(self, num_classes: int = 5, pretrained: bool = True, dropout_rate: float = 0.5):
        super(EfficientNetB4Model, self).__init__()
        weights = models.EfficientNet_B4_Weights.DEFAULT if pretrained else None
        self.model = models.efficientnet_b4(weights=weights)

        num_ftrs = self.model.classifier[1].in_features
        self.model.classifier = nn.Sequential(
            nn.Dropout(p=dropout_rate, inplace=True),
            nn.Linear(num_ftrs, num_classes),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.model(x)

    def freeze_backbone(self):
        """Standard freezing: Freezes blocks 0-3, leaves deeper blocks & classifier trainable."""
        print("Applying standard freezing strategy for EfficientNet-B4.")
        for param in self.model.parameters():
            param.requires_grad = False
        for i in range(4, 8):
            for param in self.model.features[i].parameters():
                param.requires_grad = True
        for param in self.model.classifier.parameters():
            param.requires_grad = True

    def fit(self, epoch, data_loader, optimizer, criterion, device, scheduler=None):
        self.to(device)
        self.train()
        running_loss, total, correct = 0.0, 0, 0
        
        progress_bar = tqdm.tqdm(data_loader, desc=f"Epoch {epoch+1} [TRAIN]")
        for images, labels in progress_bar:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            output = self(images)

            # Loss calculation based on ordinal type
            if criterion == "ordinal_threshold":
                num_classes_minus_1 = output.shape[1]
                targets = (labels.unsqueeze(1) > torch.arange(num_classes_minus_1, device=device)).float()
                loss = F.binary_cross_entropy_with_logits(output, targets)
                predicted = (torch.sigmoid(output) > 0.5).sum(dim=1)
            elif criterion in ["expected_value_cross_entropy", "expected_value_focal_loss"]:
                if criterion == "expected_value_focal_loss":
                    targets = F.one_hot(labels, num_classes=output.shape[1]).float()
                    base_loss = sigmoid_focal_loss(output, targets, alpha=0.25, gamma=2.0, reduction='mean')
                else:
                    weights = getattr(self, 'class_weights', None)
                    base_loss = F.cross_entropy(output, labels, weight=weights)
                
                probs = F.softmax(output, dim=1)
                class_indices = torch.arange(output.shape[1], dtype=torch.float32, device=device)
                expected_y = torch.sum(probs * class_indices, dim=1)
                ord_loss = F.smooth_l1_loss(expected_y, labels.float())
                loss = base_loss + 2.0 * ord_loss
                predicted = torch.round(expected_y).long().clamp(0, output.shape[1] - 1)
            elif criterion == "focal_loss":
                targets = F.one_hot(labels, num_classes=output.shape[1]).float()
                loss = sigmoid_focal_loss(output, targets, alpha=0.25, gamma=2.0, reduction='mean')
                _, predicted = torch.max(output.data, 1)
            else:
                loss = criterion(output, labels)
                _, predicted = torch.max(output.data, 1)

            loss.backward()
            optimizer.step()
            
            if scheduler and not isinstance(scheduler, torch.optim.lr_scheduler.ReduceLROnPlateau):
                scheduler.step()

            running_loss += loss.item() * labels.size(0)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            progress_bar.set_postfix(loss=f"{loss.item():.4f}", acc=f"{100*correct/total:.2f}%")

        return running_loss / total, 100 * correct / total

    def evaluate(self, epoch, data_loader, criterion, device):
        self.to(device)
        self.eval()
        running_loss, total, correct = 0.0, 0, 0
        all_predictions, all_labels = [], []
        
        progress_bar = tqdm.tqdm(data_loader, desc=f"Epoch {epoch+1} [VALIDATE]")
        with torch.no_grad():
            for images, labels in progress_bar:
                images, labels = images.to(device), labels.to(device)
                output = self(images)

                # Loss calculation
                if criterion == "ordinal_threshold":
                    num_classes_minus_1 = output.shape[1]
                    targets = (labels.unsqueeze(1) > torch.arange(num_classes_minus_1, device=device)).float()
                    loss = F.binary_cross_entropy_with_logits(output, targets)
                    predicted = (torch.sigmoid(output) > 0.5).sum(dim=1)
                elif criterion in ["expected_value_cross_entropy", "expected_value_focal_loss"]:
                    if criterion == "expected_value_focal_loss":
                        targets = F.one_hot(labels, num_classes=output.shape[1]).float()
                        base_loss = sigmoid_focal_loss(output, targets, alpha=0.25, gamma=2.0, reduction='mean')
                    else:
                        weights = getattr(self, 'class_weights', None)
                        base_loss = F.cross_entropy(output, labels, weight=weights)
                    probs = F.softmax(output, dim=1)
                    class_indices = torch.arange(output.shape[1], dtype=torch.float32, device=device)
                    expected_y = torch.sum(probs * class_indices, dim=1)
                    ord_loss = F.smooth_l1_loss(expected_y, labels.float())
                    loss = base_loss + 2.0 * ord_loss
                    predicted = torch.round(expected_y).long().clamp(0, output.shape[1] - 1)
                elif criterion == "focal_loss":
                    targets = F.one_hot(labels, num_classes=output.shape[1]).float()
                    loss = sigmoid_focal_loss(output, targets, alpha=0.25, gamma=2.0, reduction='mean')
                    _, predicted = torch.max(output.data, 1)
                else:
                    loss = criterion(output, labels)
                    _, predicted = torch.max(output.data, 1)

                running_loss += loss.item() * labels.size(0)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()
                
                all_labels.extend(labels.cpu().numpy())
                all_predictions.extend(predicted.cpu().numpy())
                progress_bar.set_postfix(loss=f"{loss.item():.4f}", acc=f"{100*correct/total:.2f}%")

        report = classification_report(y_true=all_labels, y_pred=all_predictions, zero_division=0)
        return running_loss / total, 100 * correct / total, report


In [21]:

class EarlyStopping:
    """Early stops the training if validation loss doesn't improve after a given patience."""
    def __init__(self, patience=7, verbose=False, delta=0, path='checkpoint.pth', trace_func=print):
        self.patience = patience
        self.verbose = verbose
        self.counter = 0
        self.best_score = None
        self.early_stop = False
        self.val_loss_min = np.inf
        self.delta = delta
        self.path = path
        self.trace_func = trace_func

    def __call__(self, val_loss, model, optimizer, scheduler, epoch):
        score = -val_loss
        if self.best_score is None:
            self.best_score = score
            self.save_checkpoint(val_loss, model, optimizer, scheduler, epoch)
        elif score < self.best_score + self.delta:
            self.counter += 1
            self.trace_func(f'EarlyStopping counter: {self.counter} out of {self.patience}')
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_score = score
            self.save_checkpoint(val_loss, model, optimizer, scheduler, epoch)
            self.counter = 0
        return self.early_stop

    def save_checkpoint(self, val_loss, model, optimizer, scheduler, epoch):
        if self.verbose:
            self.trace_func(f'Validation loss decreased ({self.val_loss_min:.6f} --> {val_loss:.6f}). Saving model...')
        checkpoint = {
            "model": model.state_dict(),
            "optimizer": optimizer.state_dict(),
            "scheduler": scheduler.state_dict(),
            "epoch": epoch,
            "val_loss": val_loss
        }
        # Atomic save to prevent corruption
        tmp_path = f"{self.path}.tmp"
        torch.save(checkpoint, tmp_path)
        if os.path.exists(tmp_path):
            os.replace(tmp_path, self.path)
        self.val_loss_min = val_loss


In [ ]:

# --- 1. Prepare Data Loaders ---
train_transform, val_transform = get_transforms(img_size=IMG_SIZE)

train_dataset = KaggleKneeOsteoarthritisDataset(root=DATASET_ROOT_PATH, split_dir="train", transform=train_transform)
train_hashes = set(train_dataset.image_hashes)

val_split_dir = "val" if os.path.isdir(os.path.join(DATASET_ROOT_PATH, "val")) else "test"
val_dataset = KaggleKneeOsteoarthritisDataset(root=DATASET_ROOT_PATH, split_dir=val_split_dir, transform=val_transform, exclude_hashes=train_hashes)

# Dynamic split if val dataset is empty after leakage removal
if len(val_dataset) == 0:
    import random
    print("Performing dynamic 80/20 train/validation split...")
    combined = list(zip(train_dataset.image_paths, train_dataset.labels))
    random.seed(42)
    random.shuffle(combined)
    split_idx = int(len(combined) * 0.8)
    train_pairs, val_pairs = combined[:split_idx], combined[split_idx:]
    
    train_dataset.image_paths, train_dataset.labels = [p for p, _ in train_pairs], [l for _, l in train_pairs]
    val_dataset.image_paths, val_dataset.labels = [p for p, _ in val_pairs], [l for _, l in val_pairs]
    print(f"Post-Split - Train: {len(train_dataset)}, Val: {len(val_dataset)}")

# Colab typically provides 2 CPU cores minimum, using 2 workers is safe
train_loader = DataLoader(dataset=train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(dataset=val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

# --- 2. Initialize Model ---
num_classes = 4 if ORDINAL_TYPE == "threshold" else 5
model = EfficientNetB4Model(num_classes=num_classes, pretrained=True)

# Calculate class weights dynamically to address class imbalance
from collections import Counter
counts = Counter(train_dataset.labels)
total_samples = sum(counts.values())
weights_list = [total_samples / (num_classes * counts[i]) if counts[i] > 0 else 1.0 for i in range(num_classes)]
class_weights = torch.tensor(weights_list, dtype=torch.float32, device=device)
model.class_weights = class_weights
print(f"Calculated class weights: {weights_list}")

# --- 3. Helper Functions for Stage setups ---
def setup_stage1(model):
    print("=== [STAGE 1] Freeze Backbone, Train Classifier Head Only ===")
    for param in model.model.parameters():
        param.requires_grad = False
    for param in model.model.classifier.parameters():
        param.requires_grad = True
    
    # Optimizer only updates classifier
    optimizer = optim.AdamW(model.model.classifier.parameters(), lr=INITIAL_LR, weight_decay=1e-2)
    return optimizer

def setup_stage2(model):
    print("=== [STAGE 2] Unfreeze Backbone & Fine-Tune Model ===")
    if not FINE_TUNE:
        model.freeze_backbone()
        optimizer = optim.AdamW(model.parameters(), lr=INITIAL_LR, weight_decay=1e-2)
    else:
        print("Applying Discriminative Fine-Tuning (3 groups) for EfficientNet-B4")
        early_backbone_params, late_backbone_params, classifier_params = [], [], []
        for n, p in model.named_parameters():
            if 'classifier' in n:
                classifier_params.append(p)
            elif 'features' in n:
                parts = n.split('.')
                try:
                    block_idx = int(parts[parts.index('features') + 1])
                    if block_idx < 4: 
                        early_backbone_params.append(p)
                    else: 
                        late_backbone_params.append(p)
                except:
                    early_backbone_params.append(p)
            else:
                early_backbone_params.append(p)
                
        optimizer = optim.AdamW([
            {'params': early_backbone_params, 'lr': INITIAL_LR * 0.01},
            {'params': late_backbone_params, 'lr': INITIAL_LR * 0.1},
            {'params': classifier_params, 'lr': INITIAL_LR}
        ], weight_decay=1e-2)
        print(f"Discriminative LRs -> Early: {INITIAL_LR * 0.01}, Late: {INITIAL_LR * 0.1}, Head: {INITIAL_LR}")
    return optimizer

# --- 4. Define Loss Criterion ---
if ORDINAL_TYPE == "threshold":
    criterion = "ordinal_threshold"
elif ORDINAL_TYPE == "expected_value":
    criterion = "expected_value_focal_loss" if USE_FOCAL_LOSS else "expected_value_cross_entropy"
else:
    criterion = "focal_loss" if USE_FOCAL_LOSS else nn.CrossEntropyLoss()

# --- 5. Define Checkpoint Paths ---
last_model_path = os.path.join(CHECKPOINT_SAVE_DIR, "last_model.pth")
best_model_stage1_path = os.path.join(CHECKPOINT_SAVE_DIR, "best_model_stage1.pth")
best_model_stage2_path = os.path.join(CHECKPOINT_SAVE_DIR, "best_model.pth")

# --- 6. Resume from Checkpoint (if exists) ---
current_stage = 1
current_epoch = 0
val_loss_min_stage1 = np.inf
val_loss_min_stage2 = np.inf
early_stop_counter_stage1 = 0
early_stop_counter_stage2 = 0

if os.path.exists(last_model_path):
    print(f"Loading local checkpoint from: {last_model_path}")
    try:
        checkpoint = torch.load(last_model_path, map_location=device)
        model.load_state_dict(checkpoint["model"])
        current_stage = checkpoint.get("stage", 1)
        current_epoch = checkpoint.get("epoch", 0) + 1
        
        if current_stage == 1:
            val_loss_min_stage1 = checkpoint.get("val_loss_min", np.inf)
            early_stop_counter_stage1 = checkpoint.get("early_stop_counter", 0)
        else:
            val_loss_min_stage2 = checkpoint.get("val_loss_min", np.inf)
            early_stop_counter_stage2 = checkpoint.get("early_stop_counter", 0)
            
        if "rng_state" in checkpoint: torch.set_rng_state(checkpoint["rng_state"].cpu())
        if "cuda_rng_state" in checkpoint and torch.cuda.is_available():
            try: torch.cuda.set_rng_state_all([s.cpu() for s in checkpoint["cuda_rng_state"]])
            except Exception: pass
        print(f"Successfully resumed from Stage {current_stage}, Epoch {current_epoch}.")
    except Exception as e:
        print(f"Could not load checkpoint ({e}). Starting from scratch.")

# --- 7. Training Loop ---

# --- STAGE 1: Train Classifier Only ---
if current_stage == 1:
    optimizer = setup_stage1(model)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=3, min_lr=1e-6)
    
    # Load optimizer state if resuming in Stage 1
    if os.path.exists(last_model_path):
        try:
            checkpoint = torch.load(last_model_path, map_location=device)
            if "optimizer" in checkpoint: optimizer.load_state_dict(checkpoint["optimizer"])
            if "scheduler" in checkpoint: 
                try: scheduler.load_state_dict(checkpoint["scheduler"])
                except Exception: pass
            for state in optimizer.state.values():
                for k, v in state.items():
                    if isinstance(v, torch.Tensor): state[k] = v.to(device)
        except Exception as e:
            print(f"Could not load Stage 1 optimizer: {e}")
            
    early_stopper = EarlyStopping(patience=5, verbose=True, path=best_model_stage1_path)
    early_stopper.val_loss_min = val_loss_min_stage1
    early_stopper.best_score = -val_loss_min_stage1
    early_stopper.counter = early_stop_counter_stage1
    
    for epoch in range(current_epoch, EPOCHS_STAGE1):
        print(f"\n--- [STAGE 1] Epoch {epoch+1}/{EPOCHS_STAGE1} ---")
        train_loss, train_acc = model.fit(epoch, train_loader, optimizer, criterion, device, scheduler)
        print(f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.2f}%")

        val_loss, val_acc, report = model.evaluate(epoch, val_loader, criterion, device)
        print(f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.2f}%")
        print(report)
        
        if isinstance(scheduler, optim.lr_scheduler.ReduceLROnPlateau): scheduler.step(val_loss)
        
        # Save last model (Atomic) for Stage 1
        checkpoint = {
            "model": model.state_dict(), 
            "optimizer": optimizer.state_dict(), 
            "scheduler": scheduler.state_dict(), 
            "epoch": epoch, 
            "stage": 1,
            "val_loss": val_loss,
            "val_loss_min": early_stopper.val_loss_min,
            "early_stop_counter": early_stopper.counter,
            "rng_state": torch.get_rng_state(),
            "cuda_rng_state": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None
        }
        tmp_path = f"{last_model_path}.tmp"
        torch.save(checkpoint, tmp_path)
        if os.path.exists(tmp_path): os.replace(tmp_path, last_model_path)
        
        if early_stopper(val_loss, model, optimizer, scheduler, epoch):
            print("Stage 1 Early stopping triggered!")
            break
            
    print("\nStage 1 finished. Loading best Stage 1 checkpoint and moving to Stage 2 fine-tuning...")
    if os.path.exists(best_model_stage1_path):
        try:
            checkpoint = torch.load(best_model_stage1_path, map_location=device)
            model.load_state_dict(checkpoint["model"])
            print("Successfully loaded best Stage 1 model weights.")
        except Exception as e:
            print(f"Could not load best Stage 1 checkpoint: {e}")
            
    # Transition to Stage 2
    current_stage = 2
    current_epoch = 0
    if os.path.exists(last_model_path):
        try: os.remove(last_model_path)
        except Exception: pass

# --- STAGE 2: Fine-Tuning ---
if current_stage == 2:
    optimizer = setup_stage2(model)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=3, min_lr=1e-6)
    
    # Load optimizer state if resuming in Stage 2
    if os.path.exists(last_model_path):
        try:
            checkpoint = torch.load(last_model_path, map_location=device)
            if "optimizer" in checkpoint: optimizer.load_state_dict(checkpoint["optimizer"])
            if "scheduler" in checkpoint: 
                try: scheduler.load_state_dict(checkpoint["scheduler"])
                except Exception: pass
            for state in optimizer.state.values():
                for k, v in state.items():
                    if isinstance(v, torch.Tensor): state[k] = v.to(device)
        except Exception as e:
            print(f"Could not load Stage 2 optimizer: {e}")
            
    early_stopper = EarlyStopping(patience=10, verbose=True, path=best_model_stage2_path)
    early_stopper.val_loss_min = val_loss_min_stage2
    early_stopper.best_score = -val_loss_min_stage2
    early_stopper.counter = early_stop_counter_stage2
    
    for epoch in range(current_epoch, EPOCHS):
        print(f"\n--- [STAGE 2] Epoch {epoch+1}/{EPOCHS} ---")
        train_loss, train_acc = model.fit(epoch, train_loader, optimizer, criterion, device, scheduler)
        print(f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.2f}%")

        val_loss, val_acc, report = model.evaluate(epoch, val_loader, criterion, device)
        print(f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.2f}%")
        print(report)
        
        if isinstance(scheduler, optim.lr_scheduler.ReduceLROnPlateau): scheduler.step(val_loss)
        
        # Save last model (Atomic) for Stage 2
        checkpoint = {
            "model": model.state_dict(), 
            "optimizer": optimizer.state_dict(), 
            "scheduler": scheduler.state_dict(), 
            "epoch": epoch, 
            "stage": 2,
            "val_loss": val_loss,
            "val_loss_min": early_stopper.val_loss_min,
            "early_stop_counter": early_stopper.counter,
            "rng_state": torch.get_rng_state(),
            "cuda_rng_state": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None
        }
        tmp_path = f"{last_model_path}.tmp"
        torch.save(checkpoint, tmp_path)
        if os.path.exists(tmp_path): os.replace(tmp_path, last_model_path)
        
        if early_stopper(val_loss, model, optimizer, scheduler, epoch):
            print("Stage 2 Early stopping triggered!")
            break


Loading 'train' split from: /content/Datasets/kaggle_knee_osteoarthritis/train

--- Dataset Statistics & Deduplication ---
  - Total files: 5778 | Unique kept: 5778
  - Internal dupes removed: 0 | Cross-split leaks removed: 0
Loading 'val' split from: /content/Datasets/kaggle_knee_osteoarthritis/val

--- Dataset Statistics & Deduplication ---
  - Total files: 826 | Unique kept: 826
  - Internal dupes removed: 0 | Cross-split leaks removed: 0
Downloading: "https://download.pytorch.org/models/efficientnet_b4_rwightman-23ab8bcd.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b4_rwightman-23ab8bcd.pth


100%|██████████| 74.5M/74.5M [00:00<00:00, 146MB/s] 


Calculated class weights: [0.5055118110236221, 1.1047801147227534, 0.7622691292875989, 1.5265521796565389, 6.679768786127168]
=== [STAGE 1] Freeze Backbone, Train Classifier Head Only ===

--- [STAGE 1] Epoch 1/20 ---


Epoch 1 [TRAIN]: 100%|██████████| 362/362 [01:14<00:00,  4.84it/s, acc=24.77%, loss=2.3837]


Train Loss: 2.9634, Train Acc: 24.77%


...

Epoch 20 [TRAIN]: 100%|██████████| 362/362 [01:18<00:00,  4.60it/s, acc=27.88%, loss=1.9362]


Train Loss: 2.3537, Train Acc: 27.88%


Epoch 20 [VALIDATE]: 100%|██████████| 52/52 [00:10<00:00,  4.95it/s, acc=27.00%, loss=3.7408]


Val Loss: 2.1651, Val Acc: 27.00%
              precision    recall  f1-score   support

           0       0.79      0.14      0.24       328
           1       0.21      0.85      0.33       153
           2       0.34      0.20      0.25       212
           3       0.31      0.04      0.07       106
           4       0.00      0.00      0.00        27

    accuracy                           0.27       826
   macro avg       0.33      0.25      0.18       826
weighted avg       0.48      0.27      0.23       826

EarlyStopping counter: 1 out of 10

--- [STAGE 2] Epoch 21/50 ---


Epoch 21 [TRAIN]:  49%|████▉     | 179/362 [00:38<00:43,  4.21it/s, acc=26.99%, loss=1.9165]